# Synthetic tabular environment

`SyntheticEnv-v1` is a custom environment bundled with mouse-env. It generates a **random finite discrete MDP** — a fully specified tabular RL problem with configurable state and action spaces. It is useful when you want a controlled environment where the exact optimal solution is known and the transition dynamics are fully available for analysis.

## Why use a synthetic env?

Real Gymnasium environments like CartPole or MountainCar have fixed structure that may not match what you want to test. `SyntheticEnv-v1` is configurable and reproducible via `seed`. Use it when:

- You want to test a tabular RL algorithm without committing to a specific env topology.
- You need ground-truth Q* values to compare against a learned policy (the `metadata_q_star` provider computes exact Q-values via value iteration).
- You want to sweep over state/action space sizes to understand how an algorithm scales.

## `metadata_q_star`: env-computed Q*

Some environments can compute Q* themselves because they have full access to the transition model. `SyntheticEnv-v1` and `Procedural-FrozenLake-v1` both support this via `q_star_source={"provider": "metadata_q_star"}`.

When this provider is active, the env runs value iteration internally whenever the MDP changes (on construction, or when a new random map is generated). The resulting Q-table is injected into every step output as `outputs[i]["q_star"]` — no external file, no separate compute step.

Constructor options (passed via `kwargs`):

| Option | Purpose |
|--------|---------|
| `obs_size` | Number of states in the MDP |
| `action_size` | Number of actions per state |

In [ ]:
from mouse_envs import EnvConfig, make_env

## Configure a random MDP

In [2]:
cfg = EnvConfig(
    id="SyntheticEnv-v1",
    seed=0,
    num_envs=2,
    max_episode_steps=50,
    kwargs={"obs_size": 8, "action_size": 3},
    q_star_source={"provider": "metadata_q_star"},
)
env = make_env(cfg)

## Inspect the stream

In [ ]:
outputs = env.step(env.sample_random_inputs())

for i, r in enumerate(outputs):
    print(
        f"env={i}  obs={r['observation'].tolist()}  "
        f"q_star_shape={r['q_star'].shape}"
    )

env.close()